# Iris Flower Classification
**A complete machine learning pipeline from raw data to model evaluation**

This notebook walks through the full workflow of building and comparing multiple classifiers on the Iris dataset using scikit-learn. The pipeline covers data loading, exploratory analysis, feature engineering, feature selection, model training, cross-validation, and final evaluation with and without the feature selection step.

**Models used:** Logistic Regression, Ridge Classifier, Random Forest, Gradient Boosting

## 1. Imports

We start by pulling in everything we need. This includes the dataset itself from scikit-learn, all the evaluation metrics, the models we plan to compare, and the usual data-science stack (pandas, numpy, matplotlib, seaborn).

In [ ]:
from sklearn.datasets import load_iris
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix,r2_score
from sklearn.model_selection import cross_val_score,train_test_split
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.linear_model import RidgeClassifier,LogisticRegression
from sklearn.feature_selection import SelectKBest,chi2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

%matplotlib inline

from warnings import filterwarnings
filterwarnings('ignore')

## 2. Loading the Data

### 2.1 Load and preview

We load the Iris dataset directly from scikit-learn and wrap it in a pandas DataFrame so it's easier to explore. The `target` column holds the class label (0, 1, or 2), each representing one of the three flower species.

In [ ]:
iris = load_iris()

df = pd.DataFrame(iris.data,columns=iris.feature_names)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 
df['target'] = iris.target

df.head()

### 2.2 Last few rows

A quick look at the tail just to confirm the data loaded completely and the class labels vary across the dataset.

In [ ]:
df.tail()

### 2.3 Column names

Listing the feature names so we know exactly what we're working with before diving deeper.

In [ ]:
df.columns

## 3. Exploratory Data Analysis

### 3.1 Statistical summary

`describe()` gives us the count, mean, standard deviation, and percentiles for every numeric feature. This is the first place where we check for unusual ranges or potential outliers.

In [ ]:
df.describe()

### 3.2 Data types and non-null counts

`info()` confirms that all columns are numeric and that there are no missing values, so we can move straight into analysis without any cleaning step.

In [ ]:
df.info()

### 3.3 Null check

A final explicit null check just to be thorough. If any column had missing values we'd need to decide how to handle them but here everything is clean.

In [ ]:
df.isna().sum()

### 3.4 Unique class labels

Checking the distinct values in the target column. We expect exactly three classes: 0 (Setosa), 1 (Versicolor), and 2 (Virginica).

In [ ]:
set(df['target'].values)

### 3.5 Class distribution

Looking at how many samples exist per class. A balanced dataset means no class will be over-represented during training, which is important for a fair comparison between models.

In [ ]:
df['target'].value_counts()

### 3.6 Target distribution plot

Visualising the class balance as a bar chart. All three classes have exactly 50 samples each, so the dataset is perfectly balanced no resampling needed.

In [ ]:
df['target'].value_counts().plot(kind='bar')
plt.title('Target Distribution')
plt.tight_layout()
plt.savefig('Data Visualization\\Target Distribution.png', dpi=300)
plt.show()

### 3.7 Correlation matrix (raw features)

Before we add any engineered features, let's see how the original four features correlate with each other and with the target. High correlation between a feature and the target is a good sign for classification.

In [ ]:
df.corr()

### 3.8 Correlation heatmap

The heatmap makes those correlations much easier to read at a glance. Petal length and petal width both show very high correlation with the target (~0.95), which tells us these two features will be the most useful for our classifiers.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')

plt.title('Correlation Matrix')
plt.tight_layout()
plt.savefig('Data Visualization\\Correlation Matrix (MAIN).png', dpi=300)

plt.show()

### 3.9 Box plot (raw features)

The box plot shows the spread and potential outliers for each feature. It also helps us see how much the value ranges differ between features useful context before feature engineering.

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(df)
plt.grid()
plt.title('Box Plot')
plt.tight_layout()
plt.savefig('Data Visualization\\Box Plot (MAIN).png', dpi=300)

plt.show()

## 4. Feature Engineering

### 4.1 Creating new features

The original dataset only gives us raw length and width measurements. We can extract more meaningful geometric information from them. For each of the sepal and petal, we calculate:

- **Area** -> length × width, a proxy for overall size
- **Ratio** -> length / width, captures the shape (elongated vs. round)
- **Diagonal** -> √(length² + width²), the approximate size of the bounding box
- **Mean** -> average of length and width
- **Difference** -> length − width, another shape descriptor

This gives us 10 new features on top of the original 4.

In [ ]:
target_names = ['setosa', 'versicolor', 'virginica']

# Define The Area from The width and Length
df['sepal area (cm2)'] = df.apply(lambda x : x['sepal length (cm)'] * x['sepal width (cm)'],axis=1)
df['petal area (cm2)'] = df.apply(lambda x : x['petal length (cm)'] * x['petal width (cm)'],axis=1)
# Define Ratio
df['sepal ratio'] = df.apply(lambda x : x['sepal length (cm)'] / x['sepal width (cm)'],axis=1)
df['petal ratio'] = df.apply(lambda x : x['petal length (cm)'] / x['petal width (cm)'],axis=1)
# Define Diagonal
df['sepal Diagonal (cm)'] = df.apply(lambda x : np.sqrt(x['sepal length (cm)']**2 + x['sepal width (cm)']**2),axis=1)
df['petal Diagonal (cm)'] = df.apply(lambda x : np.sqrt(x['petal length (cm)']**2 + x['petal width (cm)']**2),axis=1)
# Define Mean
df['sepal mean'] = df.apply(lambda x : (x['sepal length (cm)'] + x['sepal width (cm)']) / 2,axis=1)
df['petal mean'] = df.apply(lambda x : (x['petal length (cm)'] + x['petal width (cm)']) / 2,axis=1)
# Define The Difference
df['sepal difference'] = df.apply(lambda x : x['sepal length (cm)'] - x['sepal width (cm)'],axis=1)
df['petal difference'] = df.apply(lambda x : x['petal length (cm)'] - x['petal width (cm)'],axis=1)

df

### 4.2 Box plot after feature engineering

Now that we have 14 features, the box plot gives us a broader view of the distributions. The rotated x-axis labels are added here to keep the new feature names readable.

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(df)
plt.xticks(rotation=45)
plt.grid()
plt.title('Box Plot')
plt.tight_layout()
plt.savefig('Data Visualization\\Box Plot (Feature Extracted).png', dpi=300)
plt.show()

### 4.3 Correlation heatmap after feature engineering

We re-check the correlations now that the new features are in place. This helps us spot which of the engineered features are actually bringing in new information versus just being highly redundant with what we already had.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')

plt.title('Correlation Matrix')
plt.tight_layout()
plt.savefig('Data Visualization\\Correlation Matrix (Feature Extracted).png', dpi=300)
plt.show()

## 5. Feature Selection

### 5.1 Selecting the top 10 features with Chi-squared

With 14 features now available, not all of them are equally useful. We use `SelectKBest` with the chi-squared statistical test to score each feature's relationship with the target and keep only the top 10.

Chi-squared works well here because all our features are non-negative (measurements and derived geometric values), which is a requirement for the chi-squared test.

In [ ]:
X = df.drop('target',axis=1)
y = df['target']
print(X.columns)
FeatureSelection = SelectKBest(chi2,k=10)
X_Selected = FeatureSelection.fit_transform(X,y)

X_Selected_df = pd.DataFrame(X_Selected,columns=[i for i,j in zip(X.columns , FeatureSelection.get_support()) if j])
X_Selected_df

### 5.2 Correlation heatmap for selected features

A final correlation check on the selected 10 features. We want to make sure we haven't ended up with a set that's mostly redundant if two selected features are near-perfectly correlated, they're both carrying the same information.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(X_Selected_df.corr(), annot=True, cmap='coolwarm')

plt.title('Correlation Matrix')
plt.tight_layout()
plt.savefig('Data Visualization\\Correlation Matrix (Feature Selected).png', dpi=300)
plt.show()

## 6. Model Setup

### 6.1 Initialising the four classifiers

We're comparing four models that represent different approaches to classification:

- **Logistic Regression** -> a simple linear baseline, fast and interpretable
- **Ridge Classifier** -> similar to logistic regression but uses L2 regularisation to penalise large coefficients
- **Random Forest** -> an ensemble of decision trees, controlled here with `max_depth=2` and 100 trees using the Gini impurity criterion
- **Gradient Boosting** -> a sequential ensemble where each tree corrects the errors of the previous one, also capped at `max_depth=2` with 100 estimators

In [ ]:
LogisticReg = LogisticRegression()
RidgeClass = RidgeClassifier()
RandomForest = RandomForestClassifier(n_estimators=100 , criterion='gini',max_depth=2)
GradientBoostClass = GradientBoostingClassifier(n_estimators=100,max_depth=2)

## 7. Training and Evaluation - Without Feature Selection

We first train and evaluate all four models using the full engineered feature set (14 features). This gives us the baseline performance before applying feature selection.

### 7.1 Cross-validation scores (all 14 features)

Before doing a train-test split, we run 5-fold cross-validation on the complete dataset. This gives a more stable estimate of each model's generalization ability than a single split would.

In [ ]:
for model in [LogisticReg,RidgeClass,RandomForest,GradientBoostClass]:
    print(f"Model Name: {model.__class__.__name__}")
    CV_Score = cross_val_score(model,X,y,cv=5)
    print(np.round(CV_Score.mean(),3))

### 7.2 Train-test split

We split the data 80/20 - 80% for training and 20% for testing. `random_state=44` ensures the split is reproducible, and `shuffle=True` makes sure the samples are randomised before splitting (important because the dataset is sorted by class).

In [ ]:
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.20,random_state=44,shuffle=True)

#Splitted Data
print('X_train shape is ' , X_train.shape)
print('X_test shape is ' , X_test.shape)
print('y_train shape is ' , y_train.shape)
print('y_test shape is ' , y_test.shape)

### 7.3 Evaluation function

This helper function handles all the evaluation for a given model in one go. It:
1. Computes R², F1 (macro), and Accuracy
2. Produces a colour-coded bar chart - green for scores ≥ 0.7, orange for borderline, red for below 0.5
3. Plots and saves the confusion matrix
4. Prints the full per-class classification report

Saving the visuals results to `Results/No Feature Selection/` so we can compare side by side later.

In [ ]:
# Metrics Evaluation Function
def Show_Classification_Metrics(y_test,y_pred,model_name):
    metrics = {
        "R2": r2_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred, average='macro'),
        "Accuracy": accuracy_score(y_test, y_pred)
    }

    names, vals = list(metrics.keys()), list(metrics.values())
    colors = ['#4CAF50' if v >= 0.7 else '#FF7043' if v < 0.5 else '#FFA726' for v in vals]

    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(names, vals, color=colors, width=0.5, edgecolor='white', linewidth=1.2)

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{val:.4f}", ha='center', va='bottom', fontsize=12, fontweight='bold')

    ax.set_ylim(0, 1.15)
    ax.set_title(f"{model_name} - Classification Metrics", fontsize=14, pad=15)
    ax.set_ylabel("Score", fontsize=11)
    ax.axhline(0.7, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.spines[['top', 'right']].set_visible(False)

    summary = "   |   ".join([f"{k}: {v:.4f}" for k, v in metrics.items()])
    fig.text(0.5, -0.02, summary, ha='center', fontsize=11,
             bbox=dict(boxstyle='round,pad=0.4', facecolor='#f0f0f0', edgecolor='gray'))

    plt.tight_layout()
    plt.savefig(f"Results\\No Feature Selection\\{model_name} Model Evaluation.png", dpi=300, bbox_inches='tight')
    plt.show()


    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=target_names, 
                yticklabels=target_names)

    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'{model_name} - Confusion Matrix')
    plt.tight_layout()
    plt.savefig(f'Results\\No Feature Selection\\{model_name} Confusion Matrix.png', dpi=300)
    plt.show()

    print(f"{model_name} - Classification Report: \n",classification_report(y_test,y_pred,target_names=target_names))


### 7.4 Train and evaluate all models

We loop through all four models, train each one on `X_train`, make predictions on `X_test`, and call our evaluation function. The first 10 true vs. predicted labels are printed as a quick sanity check before the charts.

In [ ]:
for model in [LogisticReg,RidgeClass,RandomForest,GradientBoostClass]:
    print(f"Model Name: {model.__class__.__name__} Is Now Training")
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    print(f"Y True: \n{list(y_test[:10])}\n---\nY Predict: \n{y_pred[:10]}")
    Show_Classification_Metrics(y_test,y_pred,model.__class__.__name__)

## 8. Training and Evaluation - With Feature Selection

Now we repeat the exact same process, but this time we switch `X` to the selected 10-feature set. This lets us see whether trimming down the feature space actually helps or hurts each model.

### 8.1 Switch to the selected feature set

In [ ]:
X = X_Selected_df

### 8.2 Cross-validation scores (selected 10 features)

Same 5-fold cross-validation, now on the reduced feature set. If the scores are similar or better, it means feature selection is helping us remove noise without losing meaningful signal.

In [ ]:
for model in [LogisticReg,RidgeClass,RandomForest,GradientBoostClass]:
    print(f"Model Name: {model.__class__.__name__}")
    CV_Score = cross_val_score(model,X,y,cv=5)
    print(np.round(CV_Score.mean(),3))

### 8.3 Train-test split (selected features)

We redo the split on the new feature set using the same `random_state=44` to keep the comparison fair same test samples, just fewer input features.

In [ ]:
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.20,random_state=44,shuffle=True)

#Splitted Data
print('X_train shape is ' , X_train.shape)
print('X_test shape is ' , X_test.shape)
print('y_train shape is ' , y_train.shape)
print('y_test shape is ' , y_test.shape)

### 8.4 Updated evaluation function

The evaluation function is redefined here to save results to `Results/With Feature Selection/` — keeping these outputs separate from the previous round so we can compare both sets of charts directly.

In [ ]:
# Metrics Evaluation Function
def Show_Classification_Metrics(y_test,y_pred,model_name):
    metrics = {
        "R2": r2_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred, average='macro'),
        "Accuracy": accuracy_score(y_test, y_pred)
    }

    names, vals = list(metrics.keys()), list(metrics.values())
    colors = ['#4CAF50' if v >= 0.7 else '#FF7043' if v < 0.5 else '#FFA726' for v in vals]

    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(names, vals, color=colors, width=0.5, edgecolor='white', linewidth=1.2)

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{val:.4f}", ha='center', va='bottom', fontsize=12, fontweight='bold')

    ax.set_ylim(0, 1.15)
    ax.set_title(f"{model_name} - Classification Metrics", fontsize=14, pad=15)
    ax.set_ylabel("Score", fontsize=11)
    ax.axhline(0.7, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.spines[['top', 'right']].set_visible(False)

    summary = "   |   ".join([f"{k}: {v:.4f}" for k, v in metrics.items()])
    fig.text(0.5, -0.02, summary, ha='center', fontsize=11,
             bbox=dict(boxstyle='round,pad=0.4', facecolor='#f0f0f0', edgecolor='gray'))

    plt.tight_layout()
    plt.savefig(f"Results\\With Feature Selection\\{model_name} Model Evaluation.png", dpi=300, bbox_inches='tight')
    plt.show()


    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=target_names, 
                yticklabels=target_names)

    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'{model_name} - Confusion Matrix')
    plt.tight_layout()
    plt.savefig(f'Results\\With Feature Selection\\{model_name} Confusion Matrix.png', dpi=300)
    plt.show()

    print(f"{model_name} - Classification Report: \n",classification_report(y_test,y_pred,target_names=target_names))


### 8.5 Train and evaluate all models (with feature selection)

Final training loop same four models, same evaluation function, now running on the selected feature set. The results saved here go to `Results/With Feature Selection/` for a clean side-by-side comparison with what we got earlier.

In [ ]:
for model in [LogisticReg,RidgeClass,RandomForest,GradientBoostClass]:
    print(f"Model Name: {model.__class__.__name__} Is Now Training")
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    print(f"Y True: \n{list(y_test[:10])}\n---\nY Predict: \n{y_pred[:10]}")
    Show_Classification_Metrics(y_test,y_pred,model.__class__.__name__)

Please Leave with a vote Thank you!